# 🏆 NB-07: Reward Model Training (RLHF Style)
**Goal:** Train a reward model that scores response quality using Claude's reasoning depth as proxy.

**Modality:** Text Scoring | **Model:** DistilBERT → regression head

In [ ]:
!pip install transformers datasets torch -q

In [ ]:
import json, re, random
random.seed(42)

def load_dataset(path="claude-opus-4_5-250x_jsonl.txt"):
    """Load Claude thinking dataset from JSONL."""
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def extract_fields(records):
    data = []
    for r in records:
        msgs = r["messages"]
        user = next((m["content"] for m in msgs if m["role"]=="user"), "")
        asst = next((m["content"] for m in msgs if m["role"]=="assistant"), "")
        think = re.findall(r"<think>(.*?)</think>", asst, re.DOTALL)
        response = re.sub(r"<think>.*?</think>", "", asst, flags=re.DOTALL).strip()
        data.append({
            "user": user,
            "assistant_full": asst,
            "thinking": " ".join(think).strip(),
            "response": response
        })
    return data

records = load_dataset()
data = extract_fields(records)
print(f"Loaded {len(data)} examples")
print(f"Sample user: {data[0]['user'][:80]}")
print(f"Sample think: {data[0]['thinking'][:120]}...")
print(f"Sample resp: {data[0]['response'][:120]}...")


In [ ]:
import torch, torch.nn as nn
from transformers import DistilBertTokenizer, DistilBertModel
from torch.utils.data import Dataset as TorchDS, DataLoader
import numpy as np

# Reward proxy: log(len(thinking)) / log(len(response)) — more structured thinking = higher reward
rewards = []
for d in data:
    think_len = len(d["thinking"]) if d["thinking"] else 1
    resp_len  = max(len(d["response"]), 1)
    score = min(np.log1p(think_len) / (np.log1p(resp_len) + 1e-6), 3.0)  # normalised
    rewards.append(float(score))

print(f"Reward stats: min={min(rewards):.2f} max={max(rewards):.2f} mean={np.mean(rewards):.2f}")

tok = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

class RewardDS(TorchDS):
    def __init__(self):
        self.enc = tok([f"{d['user']} [SEP] {d['response'][:300]}" for d in data],
                       truncation=True, padding="max_length", max_length=256, return_tensors="pt")
        self.r = torch.tensor(rewards, dtype=torch.float32)
    def __len__(self): return len(self.r)
    def __getitem__(self, i): return {k: v[i] for k,v in self.enc.items()}, self.r[i]

class RewardModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.head = nn.Linear(768, 1)
    def forward(self, **kw):
        cls = self.bert(**kw).last_hidden_state[:,0,:]
        return self.head(cls).squeeze(-1)

model = RewardModel()
loader = DataLoader(RewardDS(), batch_size=16, shuffle=True)
opt = torch.optim.AdamW(model.parameters(), lr=2e-5)
loss_fn = nn.MSELoss()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

for epoch in range(4):
    total = 0
    for enc, r in loader:
        enc = {k:v.to(device) for k,v in enc.items()}
        r = r.to(device)
        pred = model(**enc)
        loss = loss_fn(pred, r)
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()
    print(f"Epoch {epoch+1} Loss: {total/len(loader):.4f}")
print("✅ Reward model trained!")
